In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Single-slide RAWH clustering on a Region of Interest (ROI)
Outputs ONLY:
  • 3×3 neighborhood tiles per cluster (ROI tiles only)
  • Original ROI crop (level-0)
  • Overlay ROI crop (level-0)
  • Per-cluster counts and fractions within ROI

ROI modes:
  - "relative": (cx,cy,w,h) in [0,1] of the whole slide at level-0
  - "manual"  : absolute level-0 box (x0,y0,x1,y1)
  - "hotspot" : attention-max window of fixed edge length at level-0
"""

from pathlib import Path
import os, gc, warnings
import numpy as np
import torch, h5py, joblib, openslide
from PIL import Image, ImageDraw
import matplotlib.colors as mcolors

# ----------- CONFIG -----------
SVS_ROOT    = Path("/common/users/wq50/CLAM/HNSCC_slides")
FEAT_DIR    = Path("/common/users/wq50/CLAM/features/HPV_UNI2_features/h5_files")
RAWH_MODEL  = Path("/common/users/wq50/CLAM2/kmeans_models/hpv_uni2_k10_rawh.joblib")
CLAM_WEIGHT = Path("/common/users/wq50/CLAM/results/HPV_CLAM_50_mb_s1/s_9_checkpoint.pt")
EMBED_DIM   = 1536
K_ASSERT    = 10

SLIDE_ID    = "TCGA-BA-6869-01Z-00-DX1.6e58648e-3309-47bb-b2c7-b71bcd9dc69b_001"  # set slide ID
# SLIDE_ID    = "TCGA-CN-6996"  # set slide ID
# SLIDE_ID    = "TCGA-CV-6951"
OUT_DIR     = Path(f"./slide_clusters_roi2_{SLIDE_ID}"); OUT_DIR.mkdir(parents=True, exist_ok=True)

TILE_SIZE_L0       = 256
NEIGHBOR_GRID      = 3
SAVE_MAX_PER_CLUSTER = None      # e.g., 120; None = save all in ROI

DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
BATCH         = 16_384

# ROI selection
ROI_MODE        = "relative"                           # "relative" | "manual" | "hotspot"
# ROI_REL         = (0.4, 0.5, 0.1, 0.1)                # if "relative": (cx_rel, cy_rel, w_rel, h_rel)
ROI_REL         = (0.5, 0.25, 0.1, 0.1)    
ROI_BOX_L0      = (10000, 12000, 16000, 18000)         # if "manual": (x0,y0,x1,y1) @ level-0
HOTSPOT_EDGE_L0 = 4096                                  # if "hotspot": edge length @ level-0

SVS_EXTS = {".svs", ".tif", ".tiff", ".ndpi", ".mrxs"}

# ----------- SAME PALETTE AS OVERLAY -----------
def base_palette_colors(K: int):
    base_hex = [
        "#E41A1C","#377EB8","#031B2E","#984EA3","#FF7F00",
        "#FFD92F","#F781BF","#66C2A5","#A65628","#999999",
    ]
    base = np.array([mcolors.to_rgb(h) for h in base_hex], dtype=float)
    reps = int(np.ceil(K / len(base_hex)))
    return (np.tile(base, (reps, 1))[:K] * 255).astype(np.uint8)

# ----------- CLAM -----------
from models.model_clam import CLAM_MB

@torch.inference_mode()
def load_clam(weight_path: Path, device: str, embed_dim: int):
    m = CLAM_MB(gate=True, size_arg="small", n_classes=2, embed_dim=embed_dim)
    sd = torch.load(weight_path, map_location=device)
    m.load_state_dict(sd, strict=False)
    return m.to(device).eval()

@torch.inference_mode()
def project_h_and_attention(block_np: np.ndarray, clam: CLAM_MB, device: str):
    x = torch.from_numpy(block_np).to(device)
    A_raw, h = clam.attention_net(x)  # A_raw: (m,2)
    a = torch.softmax(A_raw[:,0], dim=0)
    a = a * (a.numel() / a.sum())     # mean≈1
    return h.cpu().numpy().astype(np.float32), a.cpu().numpy().astype(np.float32)

# ----------- IO helpers -----------
def find_svs_by_stem(root: Path, stem: str) -> Path | None:
    for p in root.rglob("*"):
        if p.suffix.lower() in SVS_EXTS and p.stem == stem:
            return p
    return None

def iter_h5(h5_path: Path, batch=BATCH):
    with h5py.File(h5_path, "r") as f:
        X = f["features"]; C = f["coords"]
        N = X.shape[0]
        for off in range(0, N, batch):
            yield off, X[off:off+batch][:].astype(np.float32), C[off:off+batch][:].astype(np.int32)

# ----------- Clustering -----------
def assign_and_dist_h(H: np.ndarray, km) -> tuple[np.ndarray, np.ndarray]:
    labs = km.predict(H)
    C = km.cluster_centers_.astype(np.float32)
    d = np.linalg.norm(H - C[labs], axis=1)
    return labs.astype(np.int32), d.astype(np.float32)

# ----------- Tile reading -----------
def read_tile(slide, x0, y0, edge):
    region = slide.read_region((x0, y0), 0, (edge, edge))
    rgba = region.convert("RGBA")
    bg = Image.new("RGBA", rgba.size, (255,255,255,255))
    return Image.alpha_composite(bg, rgba).convert("RGB")

def extract_neighborhood_grid(svs_path: Path, x0: int, y0: int,
                              tile_edge: int = TILE_SIZE_L0, grid: int = 3) -> Image.Image:
    slide = openslide.OpenSlide(str(svs_path))
    step = tile_edge; half = grid // 2
    tiles = []
    for dy in range(-half, half+1):
        row = []
        for dx in range(-half, half+1):
            xi = x0 + dx*step; yi = y0 + dy*step
            row.append(read_tile(slide, xi, yi, step))
        tiles.append(row)
    slide.close()
    w, h = tiles[0][0].width, tiles[0][0].height
    canvas = Image.new("RGB", (grid*w, grid*h), (255,255,255))
    for r in range(grid):
        for c in range(grid):
            canvas.paste(tiles[r][c], (c*w, r*h))
    return canvas

# ----------- ROI utilities -----------
def rel_box_to_abs_level0(rel_box, slide_wh_l0, align=TILE_SIZE_L0):
    cxr, cyr, wr, hr = rel_box
    W0, H0 = slide_wh_l0
    wr = float(np.clip(wr, 0.0, 1.0))
    hr = float(np.clip(hr, 0.0, 1.0))
    cx = float(np.clip(cxr, 0.0, 1.0)) * W0
    cy = float(np.clip(cyr, 0.0, 1.0)) * H0
    w = max(align, int(round(wr * W0)))
    h = max(align, int(round(hr * H0)))
    x0 = int(round(cx - w/2)); y0 = int(round(cy - h/2))
    x1 = x0 + w; y1 = y0 + h
    # clamp
    x0 = max(0, min(x0, W0 - 1)); y0 = max(0, min(y0, H0 - 1))
    x1 = max(1, min(x1, W0));     y1 = max(1, min(y1, H0))
    # re-fit if overflow after clamp
    if x1 - x0 < w: x0 = max(0, x1 - w)
    if y1 - y0 < h: y0 = max(0, y1 - h)
    # snap to tile grid
    x0 = (x0 // align) * align
    y0 = (y0 // align) * align
    x1 = min(W0, x0 + ((w // align) * align))
    y1 = min(H0, y0 + ((h // align) * align))
    if x1 <= x0: x1 = min(W0, x0 + align)
    if y1 <= y0: y1 = min(H0, y0 + align)
    return (x0, y0, x1, y1)

def choose_hotspot_roi(coords_l0: np.ndarray, attn: np.ndarray,
                       edge_l0: int, slide_wh_l0: tuple[int,int]) -> tuple[int,int,int,int]:
    W, H = slide_wh_l0
    tx = coords_l0[:,0] // TILE_SIZE_L0
    ty = coords_l0[:,1] // TILE_SIZE_L0
    gx = int(np.ceil(W / TILE_SIZE_L0)); gy = int(np.ceil(H / TILE_SIZE_L0))
    dens = np.zeros((gy, gx), dtype=np.float32)
    np.add.at(dens, (ty, tx), attn.astype(np.float32))
    win = max(1, int(round(edge_l0 / TILE_SIZE_L0)))
    integ = dens.cumsum(0).cumsum(1)
    def rect_sum(x0,y0,x1,y1):
        s = integ[y1,x1]
        if x0>0: s -= integ[y1,x0-1]
        if y0>0: s -= integ[y0-1,x1]
        if x0>0 and y0>0: s += integ[y0-1,x0-1]
        return s
    best = (-1, 0, 0)
    for y in range(0, gy - win + 1):
        y1 = y + win - 1
        for x in range(0, gx - win + 1):
            x1 = x + win - 1
            val = rect_sum(x, y, x1, y1)
            if val > best[0]:
                best = (val, x, y)
    _, bx, by = best
    x0 = bx * TILE_SIZE_L0; y0 = by * TILE_SIZE_L0
    return (x0, y0, x0 + edge_l0, y0 + edge_l0)

def filter_by_roi(coords_l0: np.ndarray, roi_box_l0: tuple[int,int,int,int]) -> np.ndarray:
    x0,y0,x1,y1 = roi_box_l0
    return ((coords_l0[:,0] >= x0) & (coords_l0[:,0] < x1) &
            (coords_l0[:,1] >= y0) & (coords_l0[:,1] < y1))

# ----------- Overlay directly on ROI at level-0 -----------
def render_roi_overlay_level0(svs_path: Path,
                              roi_box_l0: tuple[int,int,int,int],
                              coords_roi_l0: np.ndarray,
                              labels_roi: np.ndarray,
                              colors_uint8: np.ndarray,
                              tile_size_l0: int = TILE_SIZE_L0,
                              alpha: int = 120) -> tuple[Image.Image, Image.Image]:
    """
    Returns (roi_original_rgb, roi_overlay_rgb) at level-0 resolution.
    """
    x0, y0, x1, y1 = roi_box_l0
    edge_w, edge_h = x1 - x0, y1 - y0

    slide = openslide.OpenSlide(str(svs_path))
    region = slide.read_region((x0, y0), 0, (edge_w, edge_h))  # RGBA
    slide.close()

    rgba = region.convert("RGBA")
    bg = Image.new("RGBA", rgba.size, (255,255,255,255))
    roi_rgb = Image.alpha_composite(bg, rgba).convert("RGB")

    overlay = Image.new("RGBA", (edge_w, edge_h), (0,0,0,0))
    draw = ImageDraw.Draw(overlay, "RGBA")

    w = tile_size_l0
    # draw ROI tiles offset to ROI origin
    for (tx, ty), lab in zip(coords_roi_l0, labels_roi):
        x = int(tx - x0); y = int(ty - y0)
        r,g,b = map(int, colors_uint8[int(lab)])
        draw.rectangle([x, y, x+w, y+w], fill=(r, g, b, alpha), outline=None)

    roi_overlay_rgb = Image.alpha_composite(roi_rgb.convert("RGBA"), overlay).convert("RGB")
    return roi_rgb, roi_overlay_rgb

# ----------- Main -----------
def main():
    h5 = FEAT_DIR / f"{SLIDE_ID}.h5"
    svs = find_svs_by_stem(SVS_ROOT, SLIDE_ID)
    assert h5.exists(), f"H5 not found: {h5}"
    assert svs is not None and svs.exists(), f"SVS not found for: {SLIDE_ID}"

    km = joblib.load(RAWH_MODEL)
    assert km.n_clusters == K_ASSERT, f"Expected k={K_ASSERT}"
    clam = load_clam(CLAM_WEIGHT, DEVICE, EMBED_DIM)

    # stream features to get h and attention
    H_list, C_list, A_list = [], [], []
    for _, X, C in iter_h5(h5, BATCH):
        H, a = project_h_and_attention(X, clam, DEVICE)
        H_list.append(H); C_list.append(C); A_list.append(a)
        del H, X; gc.collect()
    H_all   = np.concatenate(H_list, axis=0).astype(np.float32)
    coords  = np.concatenate(C_list, axis=0).astype(np.int32)
    attn    = np.concatenate(A_list, axis=0).astype(np.float32)

    # slide dimensions
    slide = openslide.OpenSlide(str(svs))
    W0, H0 = slide.dimensions
    slide.close()

    # pick ROI
    if ROI_MODE == "relative":
        roi_box = rel_box_to_abs_level0(ROI_REL, (W0, H0), align=TILE_SIZE_L0)
    elif ROI_MODE == "manual":
        roi_box = ROI_BOX_L0
    elif ROI_MODE == "hotspot":
        roi_box = choose_hotspot_roi(coords, attn, HOTSPOT_EDGE_L0, (W0, H0))
    else:
        raise ValueError("ROI_MODE must be 'relative', 'manual', or 'hotspot'")

    in_roi = filter_by_roi(coords, roi_box)
    if not np.any(in_roi):
        raise RuntimeError("No tiles in ROI. Adjust ROI params.")

    # restrict to ROI and cluster
    H = H_all[in_roi]
    coords_roi = coords[in_roi]
    labels, dists = assign_and_dist_h(H, km)

    # outputs
    base_dir = OUT_DIR / f"{SLIDE_ID}"
    (base_dir / "overlay").mkdir(parents=True, exist_ok=True)
    for k in range(km.n_clusters):
        (base_dir / f"cluster_{k:02d}").mkdir(parents=True, exist_ok=True)

    # counts and fractions
    counts = np.bincount(labels, minlength=km.n_clusters).astype(int)
    frac = counts / max(1, counts.sum())
    np.save(base_dir / "roi_cluster_counts.npy", counts)
    np.save(base_dir / "roi_cluster_fracs.npy", frac)

    # also a small TSV for convenience
    import pandas as pd
    pd.DataFrame({"cluster": np.arange(km.n_clusters), "count": counts, "frac": frac}) \
      .to_csv(base_dir / "roi_cluster_summary.tsv", sep="\t", index=False)

    # save 3×3 neighborhoods per cluster for ROI tiles only
    per_cluster_written = [0]*km.n_clusters
    for i, ((x,y), lab, dist) in enumerate(zip(coords_roi, labels, dists)):
        if SAVE_MAX_PER_CLUSTER is not None and per_cluster_written[lab] >= SAVE_MAX_PER_CLUSTER:
            continue
        grid = extract_neighborhood_grid(svs, int(x), int(y), TILE_SIZE_L0, NEIGHBOR_GRID)
        p = base_dir / f"cluster_{lab:02d}" / f"roi_nbr3x3_tile{i:06d}_x{x}_y{y}_d{dist:.4f}.png"
        grid.save(p, format="PNG", optimize=True)
        per_cluster_written[lab] += 1

    # render ORIGINAL ROI and OVERLAY ROI at level-0
    colors_uint8 = base_palette_colors(km.n_clusters)
    roi_orig_rgb, roi_overlay_rgb = render_roi_overlay_level0(
        svs_path=svs,
        roi_box_l0=roi_box,
        coords_roi_l0=coords_roi,
        labels_roi=labels,
        colors_uint8=colors_uint8,
        tile_size_l0=TILE_SIZE_L0,
        alpha=120
    )
    roi_orig_rgb.save(base_dir / "overlay" / f"{SLIDE_ID}_ROI_original.png", quality=95)
    roi_overlay_rgb.save(base_dir / "overlay" / f"{SLIDE_ID}_ROI_overlay.png", quality=95)

    # console summary
    print(f"ROI (L0): {roi_box}")
    print("Counts:", counts.tolist())
    print("Fracs :", np.round(frac, 3).tolist())
    print("3x3 tiles saved per cluster:", per_cluster_written)
    print("Done →", base_dir.resolve())

if __name__ == "__main__":
    warnings.filterwarnings("ignore", message=".*get_cmap function was deprecated.*")
    torch.set_float32_matmul_precision("high")
    os.environ["PYTHONHASHSEED"] = "1337"
    np.random.seed(1337); torch.manual_seed(1337)
    main()

ROI (L0): (34304, 11264, 41728, 16896)
Counts: [171, 1, 85, 37, 0, 327, 0, 0, 2, 15]
Fracs : [0.268, 0.002, 0.133, 0.058, 0.0, 0.513, 0.0, 0.0, 0.003, 0.024]
3x3 tiles saved per cluster: [171, 1, 85, 37, 0, 327, 0, 0, 2, 15]
Done → /common/users/wq50/CLAM2/slide_clusters_roi2_TCGA-BA-6869-01Z-00-DX1.6e58648e-3309-47bb-b2c7-b71bcd9dc69b_001/TCGA-BA-6869-01Z-00-DX1.6e58648e-3309-47bb-b2c7-b71bcd9dc69b_001
